# 1. 块
块可以理解为单个层、多个层构成的组件或者整个模型，使用块进行抽象有一个好处就是可以将一些块组成一个更大的组件，这个过程通常是递归形式的

从编程角度来看，块由class定义，由于自动微分，所以在自定义类时只需要考虑**前向传播函数和必需参数**

## 自定义块
在进行自定义块之前，我们先回顾一下**多层感知机**的代码实现

In [24]:
import torch
from torch import nn
from torch.nn import functional as F
net = nn.Sequential(nn.Flatten(), 
                nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
                nn.Linear(256, 10))
X = torch.rand(2, 28, 28)
print(net(X))

tensor([[ 0.0911,  0.0383, -0.0583,  0.0178, -0.0143,  0.0858,  0.1451,  0.0597,
         -0.0004, -0.0984],
        [ 0.1582, -0.0879, -0.1292,  0.1070, -0.1124, -0.0422,  0.1497,  0.1016,
          0.0272,  0.0004]], grad_fn=<AddmmBackward0>)


我们通过**实例化nn.Sequential**来构建模型，nn.Sequential定义了一种特殊的Module，即在PyTorch中表示一个**块**的类，它维护一个由Module组成的有序列表。注意，三个全连接层都是Linear类的实例，Linear类本身就是Module的子类

`net(X)`调用模型获得模型的输出，实际上它是`net.__call__(X)`的简写

### 每个块必须提供的基本功能：
* **前向传播函数接收输入数据作为参数**

`forward(X)`，模块拿到张量输入，所有运算基于输入 `X`。

* **forward 生成输出，输入、输出形状可以不一致**

线性层、卷积层都会改变张量维度；比如 `Linear(784,256)`，784 维进，256 维出。

* **支持反向传播，自动计算输出对参数的梯度**

前向算预测值；loss 反向传播时，框架自动求出梯度，存在 `.grad`。你不用手写求导。

* **能够保存、读取自身可训练参数（权重 weight、偏置 bias）**

`nn.Linear` 内部自带 `self.weight, self.bias`，模块自己持有参数，方便优化器更新。

* **提供参数初始化机制**

创建层时可以初始化权重；也允许你手动覆盖初始化策略。

### 自定义块

In [25]:
class MLP(nn.Module): # 继承pytorch中定义块的类
    def __init__(self):
        """用模型参数声明层"""
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

注意
* 我们定制的`__init__`函数通过`super().__init__()`调用父类的`__init__`函数，省去了重复编写模版代码的痛苦。
* 除非我们需要实现一个新的运算符，否则我们不必担心反向传播函数或参数初始化，系统将自动生成这些

In [26]:
net = MLP()
X = torch.rand(2, 20)
net(X)

tensor([[ 0.0170, -0.0567, -0.0141,  0.1554, -0.2083,  0.2212,  0.0116,  0.0594,
         -0.1719, -0.0653],
        [ 0.0259, -0.0152, -0.0996,  0.0400, -0.1164,  0.0426, -0.0268,  0.0320,
         -0.0976,  0.0364]], grad_fn=<AddmmBackward0>)

## 顺序块
为了构建我们自己的简化的`MySequential`，
我们只需要定义两个关键函数：
* 一个可以将块逐个追加到列表中的函数
* 一个前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”

In [27]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # _modules是父类自带的特殊有序字典OrderedDict
            self._modules[str(idx)] = module
    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [28]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.2430,  0.3613, -0.0488,  0.0311, -0.1365, -0.0935,  0.0443,  0.0767,
          0.0568,  0.0572],
        [ 0.1076,  0.2447,  0.0222,  0.0332, -0.1078, -0.0397,  0.0832,  0.1539,
          0.1046,  0.0064]], grad_fn=<AddmmBackward0>)

## 在前向传播函数中执行代码
到目前为止，我们网络中的所有操作都对网络的激活值及网络的参数起作用。然而，有时我们可能希望合并既不是上一层的结果也不是可更新参数的项，
我们称之为*常数参数*（constant parameter）。
例如，我们需要一个计算函数
$f(\mathbf{x},\mathbf{w}) = c \cdot \mathbf{w}^\top \mathbf{x}$的层，
其中$\mathbf{x}$是输入，
$\mathbf{w}$是参数，
$c$是某个在优化过程中没有更新的指定常量。

In [29]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

在这个`FixedHiddenMLP`模型中，我们实现了一个隐藏层，
其权重（`self.rand_weight`）在实例化时被随机初始化，之后为常量。
这个权重不是一个模型参数，因此它永远不会被反向传播更新。
然后，神经网络将这个固定层的输出通过一个全连接层。

注意，在返回输出之前，模型做了一些不寻常的事情：
它运行了一个while循环，在$L_1$范数大于$1$的条件下，
将输出向量除以$2$，直到它满足条件为止。
最后，模型返回了`X`中所有项的和。
注意，此操作可能不会常用于在任何实际任务中，
我们只展示如何将任意代码集成到神经网络计算的流程中。

In [30]:
net = FixedHiddenMLP()
net(X)

tensor(0.2012, grad_fn=<SumBackward0>)

## 混合块

In [31]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.0679, grad_fn=<SumBackward0>)

## 问题
### 如果将`MySequential`中存储块的方式更改为Python列表，会出现什么样的问题？

`net.parameters()` / `net.named_parameters()` **找不到 Linear 的 weight、bias**；

优化器无法获取权重，训练时参数不会更新；

模型相当于 “只有前向能跑，但无法训练”。

`.to(device)` / `.cuda()` / `.cpu()` 不会自动迁移列表内模块
### 平行块实现

In [32]:
class ParallelBlock(nn.Module):
    def __init__(self, net1: nn.Module, net2: nn.Module):
        super().__init__()
        self.net1 = net1
        self.net2 = net2
    def forward(self, X):
        out1 = self.net1(X)
        out2 = self.net2(X)
        return torch.cat((out1, out2), dim = 1)

### 生成同一个块的多个实例

In [33]:
def make_multi_block(block, num_blocks):
    """
    block：无参函数，调用可以生成一个网络块实例
    num_blocks：重复块数量
    返回：组装好的 nn.Sequential
    """
    layers = []
    for _ in range(num_blocks):
        layers.append(block())
    return nn.Sequential(*layers)
def get_block():
    return nn.Sequential(nn.Linear(32, 32), nn.ReLU())
net = make_multi_block(get_block, 4)
X = torch.rand(2, 32)
print(net(X))

tensor([[0.0314, 0.0000, 0.0000, 0.0000, 0.0258, 0.1006, 0.1677, 0.0000, 0.1213,
         0.0000, 0.0000, 0.0000, 0.1680, 0.0743, 0.0335, 0.0000, 0.0000, 0.1487,
         0.0000, 0.0494, 0.0000, 0.0862, 0.1904, 0.0130, 0.1489, 0.0626, 0.1254,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0422, 0.0000, 0.0000, 0.0000, 0.0512, 0.0958, 0.2043, 0.0000, 0.1315,
         0.0000, 0.0236, 0.0000, 0.1692, 0.0779, 0.0191, 0.0000, 0.0000, 0.1682,
         0.0000, 0.0859, 0.0000, 0.0710, 0.1931, 0.0324, 0.1815, 0.0379, 0.1293,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)


# 2. 参数管理

## 2.1 参数访问
先构建一个简单的MLP

In [34]:
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(2, 4)
print(net(X))

tensor([[0.1040],
        [0.0864]], grad_fn=<AddmmBackward0>)


当我们用Sequential类来定义模型时，我们可以通过**索引**来访问模型的任意层，比如`net[0]`表示`nn.Linear(20, 256)`，`net[1]`表示`nn.ReLU()`

`.state_dict()`返回**该模块或者整个模型**的**所有参数和buffer**的有序字典

区分参数和buffer的特征：

* `model.parameters()`或`model.named_parameters()`：默认`requires_grad=True`，参与梯度计算，优化器更新
* `model.register_buffer`，`model.named_buffers()`：不在`parameters()`里面，优化器不会更新，跟随模型`.to(device)`,`.cuda()`自动迁移设备

In [35]:
for name, param in net.named_parameters():
    print(name)
for name, buf in net.named_buffers():
    print(name)
print(net.state_dict())

0.weight
0.bias
2.weight
2.bias
OrderedDict([('0.weight', tensor([[ 0.0049,  0.3915, -0.2009, -0.4064],
        [-0.1176,  0.1282, -0.3420,  0.3875],
        [-0.2140,  0.4403,  0.0941,  0.0530],
        [-0.4850, -0.2387,  0.2714,  0.2770],
        [ 0.0133,  0.3386, -0.4077, -0.1284],
        [-0.2490, -0.4761, -0.3464, -0.1040],
        [ 0.3457,  0.1159, -0.2893,  0.1724],
        [ 0.4756, -0.4414,  0.4723,  0.2638]])), ('0.bias', tensor([-0.3191, -0.3084,  0.4942, -0.2019, -0.2084, -0.0284,  0.1237,  0.3302])), ('2.weight', tensor([[ 0.3119, -0.2127, -0.1907, -0.0570, -0.2010, -0.2225,  0.3232,  0.0311]])), ('2.bias', tensor([0.2081]))])


### 目标参数
每个参数都表示为参数类的一个实例

In [36]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)
net[2].weight.grad == None

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2081], requires_grad=True)
tensor([0.2081])


True

### 一次性访问所有参数
当我们处理更复杂的块时（比如嵌套块），情况可能变得特别复杂，因此我们需要**递归整个树**来提取每个子块的参数
* `.parameters()`返回仅参数张量的迭代器
* `.named_parameters()`返回(名称字符串，参数张量)二元组迭代器

In [37]:
print(*((name, param.shape) for name, param in net[0].named_parameters()))
print(*((name, param.shape) for name, param in net.named_parameters()))

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [38]:
print(net.state_dict()["2.bias"])

tensor([0.2081])


### 嵌套块

`.add_module(name, module)` 注册子模块，等价于`._modules[name] = module`

In [39]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4))
def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i}', block1())
    return net
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
print(rgnet(X))

tensor([[0.3840],
        [0.3837]], grad_fn=<AddmmBackward0>)


In [40]:
print(rgnet) # 调用__repr__方法

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [41]:
print(rgnet[0][1][0])

Linear(in_features=4, out_features=8, bias=True)


## 2.2 参数初始化

### 内置初始化
`.apply()`
**递归遍历所有层级内的每一个子模块**,不必担心嵌套块的问题
#### 高斯随机初始化

In [42]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean = 0, std = 0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0123, -0.0158,  0.0036,  0.0084]), tensor(0.))

#### 常数初始化

In [43]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

当然我们也可以对不同层应用不同参数初始化方法

In [44]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)
net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.1292, -0.3597,  0.0547, -0.5364])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### nn.init常用原地初始化函数
https://docs.pytorch.org/docs/2.13/nn.init.html
#### 1. 均匀分布

```
uniform_(tensor, a=0.0, b=1.0)
```

均匀分布 \(U[a,b]\)

```
xavier_uniform_(tensor, gain=1.0)
```

Xavier 均匀初始化（适合 sigmoid/tanh），gain是缩放系数，用来适配激活函数，人为放大/缩小初始化权重的取值范围
| 激活函数   | gain 值        |
| ---------- | -------------- |
| tanh       | 1.0            |
| sigmoid    | 1.0            |
| relu       | $\sqrt{2}$ ≈1.414        |
| leaky_relu | 根据负斜率计算 |

```
kaiming_uniform_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')
```

He 均匀初始化（ReLU 网络首选）

#### 2. 正态分布

```
normal_(tensor, mean=0.0, std=1.0)
```

标准正态分布

```
xavier_normal_(tensor, gain=1.0)
```

Xavier 正态初始化

```
kaiming_normal_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')
```

He 正态初始化



### 自定义初始化
有时，深度学习框架没有提供我们需要的初始化方法。
在下面的例子中，我们使用以下的分布为任意权重参数$w$定义初始化方法：

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \text{ 可能性 } \frac{1}{4} \\
            0    & \text{ 可能性 } \frac{1}{2} \\
        U(-10, -5) & \text{ 可能性 } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


In [45]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-0.0000,  9.9338, -0.0000,  0.0000],
        [-5.4387,  9.6920,  0.0000,  8.7469]], grad_fn=<SliceBackward0>)

注意，我们始终可以直接设置参数。

In [46]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000, 10.9338,  1.0000,  1.0000])

## 2.3 参数绑定

In [47]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


当参数绑定时，梯度会发生什么变化？

两个不同位置使用同一套绑定参数，反向传播时，两条路径传回该权重的梯度**自动相加**；
优化器使用**累加后的总梯度**更新唯一一份权重。

## 2.4 延后初始化机制
直到数据第一次通过模型传递时，框架才会动态判断出每个层的大小

详细解释：
* 实例化网络时，**不预先指定输入维度**，框架暂不知道权重矩阵形状
* 因此，还没有**创建、初始化任何参数**
* **第一次执行前向传播**，框架读取输入维度，自上而下逐层推断每一层参数的shape
* 推断完成后，**一次性创建并初始化所有权重参数**


In [48]:
from torch.nn import LazyLinear
# 只写输出维度，输入维度留空，且创建时不生成权重
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(),
                    nn.LazyLinear(10))
# ========= 此时：还没有参数！访问weight会报错 =========
# print(net[0].weight)  # 报错！参数尚未创建
# ========= 送入真实数据，触发延后初始化 =========
X = torch.rand(2,20)
y = net(X)
# ========= 现在参数已经自动创建完成 =========
print(net[0].weight.shape)   
print(net[2].weight.shape)  

torch.Size([256, 20])
torch.Size([10, 256])


c:\Users\Administrator\Anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


缺点：
* LazyLinear、LazyConv2d这类延后初始化模块属于试验性功能，不保证接口稳定
* 必须运行一次前向传播才能看到参数
* 无法在实例化网络之后、喂数据之前手动执行自定义初始化

# 3. 自定义层
## 3.1 不带参数的层
下面的`CenteredLayer`类要从其输入中减去均值。
要构建它，我们只需继承基础层类并实现前向传播功能。

In [51]:
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, X):
        return X - X.mean()

让我们向该层提供一些数据，验证它是否能按预期工作。

In [52]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

现在，我们可以[**将层作为组件合并到更复杂的模型中**]。

In [53]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

作为额外的健全性检查，我们可以在向该网络发送随机数据后，检查均值是否为0。
由于我们处理的是浮点数，因为存储精度的原因，我们仍然可能会看到一个非常小的非零数。

In [54]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(-3.7253e-09, grad_fn=<MeanBackward0>)

## 3.2 带参数的层
我们可以使用内层函数来创建参数，这些函数提供一些基本的管理功能。比如管理**访问、初始化、共享、保存和加载模型参数**，不需要为每个自定义层编写自定义的序列化程序

In [61]:
class MyLinearRelu(nn.Module):
    def __init__(self, in_units, out_units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, out_units))
        self.bias = nn.Parameter(torch.rand(out_units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight) + self.bias
        return F.relu(linear)

接下来，我们实例化`MyLinear`类并访问其模型参数。

In [62]:
linear = MyLinearRelu(5, 3)
linear.weight

Parameter containing:
tensor([[ 0.1037,  1.8463,  0.0423],
        [-1.2424,  0.9015,  0.4598],
        [-0.4160,  0.4415,  1.4249],
        [-0.2373, -0.1292, -0.3152],
        [ 1.2303,  0.2316,  1.1705]], requires_grad=True)

我们可以[**使用自定义层直接执行前向传播计算**]。

In [65]:
linear(torch.rand(2, 5))

tensor([[0.0000, 1.8739, 2.1547],
        [0.0000, 1.5413, 2.3665]], grad_fn=<ReluBackward0>)

## 二次型
设计一个接受输入并计算张量降维的层，它返回$y_k = \sum_{i, j} W_{ijk} x_i x_j$。

In [66]:
class QuadraticReduction(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        # 权重 W_{ijk}: [i, j, k]
        self.W = nn.Parameter(torch.randn(in_dim, in_dim, out_dim))

    def forward(self, x):
        # x: (batch_size, in_dim)
        batch_size = x.shape[0]
        # 方式1：高效广播实现求和 ∑_{i,j} W_{ijk} x_i x_j
        # x_i x_j : (B, d, 1) * (B, 1, d) = (B, d, d)
        outer = x.unsqueeze(2) * x.unsqueeze(1)   # (B, d, d)
        # outer (B,d,d)  @ W (d,d,K) -> (B,K)
        y = torch.einsum("bij,ijk->bk", outer, self.W)
        return y

In [67]:
layer = QuadraticReduction(in_dim=3, out_dim=2)
X = torch.randn(4, 3)   # batch=4, 输入3维
Y = layer(X)
print(Y.shape)   # torch.Size([4, 2])

torch.Size([4, 2])


## 傅里叶变换
设计一个返回输入数据的傅立叶系数前半部分的层。

# 4. 读写文件
## 4.1 加载和保存向量
`torch.save(obj, f)`
序列化保存张量/模型/任意对象到磁盘，底层使用Python pickle序列化

`obj`:张量、模型、列表、字典都可以保存

`f`:文件名字符串，运行后在当前目录生成文件，没有后缀名

`torch.load(f, map_location = None, pickle_model = pickle, **pickle_load_args)` 
读取由torch.save()序列化保存的文件，反序列化恢复对象

`map_location`：控制张量加载到哪个设备

In [82]:
x = torch.arange(4)
import os
fp = os.path.join('..', 'models')
os.makedirs(fp, exist_ok = True)
f = os.path.join('..', 'models', 'x-file.pth')
torch.save(x, f)

In [83]:
x2 = torch.load(f, map_location = "cuda:0")
print(x2)

tensor([0, 1, 2, 3], device='cuda:0')


In [76]:
y = torch.zeros(4)
torch.save([x, y],f)
x2, y2 = torch.load(f)
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我们甚至可以(**写入或读取从字符串映射到张量的字典**)。
当我们要读取或写入模型中的所有权重时，这很方便。

In [77]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## 加载和保存模型参数

In [78]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

接下来，我们[**将模型的参数存储在一个叫做“mlp.params”的文件中。**]

In [79]:
torch.save(net.state_dict(), 'mlp.params')

为了恢复模型，我们[**实例化了原始多层感知机模型的一个备份。**]
这里我们不需要随机初始化模型参数，而是(**直接读取文件中存储的参数。**)

In [84]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params', map_location = "cuda:0"))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

切换到eval模式
* Dropout：关闭，不再随机丢弃神经元，
* BatchNorm：使用训练阶段统计的均值方差，不再实时更新均值方差

## 复用网络的一部分
```python
# 原有网络（举例：3层MLP）
class OldMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 30)
        self.layer3 = nn.Linear(30, 5)

# 新网络：复用 layer1、layer2，新增自定义头部（替换第三层）
class NewMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 30)
        self.new_head = nn.Linear(30, 8)  # 全新层

# 1. 实例化新网络
new_net = NewMLP()

# 2. 加载旧模型全部参数
old_state_dict = torch.load("old.params")
new_state_dict = new_net.state_dict()

# 3. 筛选、拷贝前两层匹配参数
transfer_keys = ["layer1.weight", "layer1.bias", "layer2.weight", "layer2.bias"]
for key in transfer_keys:
    if key in old_state_dict and key in new_state_dict:
        new_state_dict[key] = old_state_dict[key]

# 4. 载入新网络
new_net.load_state_dict(new_state_dict)
```
若新旧层的名字不一样,手动写映射
```python
mapping = {
    "linear1.weight": "backbone1.weight",
    "linear1.bias": "backbone1.bias",
    "linear2.weight": "backbone2.weight",
    "linear2.bias": "backbone2.bias"
}
for old_key, new_key in mapping.items():
    new_state_dict[new_key] = old_state_dict[old_key]
```

## 保存整个模型
需要遵守：
* 加载环境中必须能找到这个网络类的定义
* 不要修改类名、类结构、导入路径
* 兼容性差，不同pytorch版本可能加载失败